# 01b - Can the J-lens read a confidence number before it is spoken?

**Hours 3-4. Instrument validation, gate V1b.**

V1 surfaced `" Judy"` -- a name copied out of the context. PLAN.md 4 assumes
something harder: that a verbalized *confidence number* is represented at the
answer-adjacent position before it is emitted (inherited from 2604.01457).
Nothing has checked that. If it is false, 5.3-5.5 are unreadable.

> **Not a sixth prompt condition.** The P1-P5 grid in PLAN.md 5.1 is fixed (R9).
> These are synthetic items with ground truth we control, used to test the
> *instrument*. Nothing here feeds the pre-registered measurement.

## The 0-100 scale

Confidence is a **whole number from 0 to 100** -- the format the reference
literature uses (2603.25052), so the numbers here are directly comparable to
published ones.

That comparability is not free. A 0-100 value is **not a single token**: `85` is
two slots, `100` is three. Two consequences run through everything below:

- the confidence *distribution* is a **sequence score** over candidate values
  (`synthetic.score_candidates`, 0-100 in steps of 5), not one softmax row. That
  is 21 short forward passes per item instead of one, and it is the defensible
  version -- a single-position argmax over digits would score `8` and call it 85.
- the J-lens reads **position `-1` only**, which is the number's **first digit**.
  So the S1 readout test is a first-digit test: `85` and `82` are the same event
  to it. Section 4 is where the *whole* number is read, one slot at a time, and
  on this scale it is no longer optional.

The single-token 1-9 scale is still available via `SCALE = "nine"`, where the
lens readout and the final distribution sit at the identical slot. Section 4's
first-digit-vs-whole-number comparison is the evidence for which scale 02 should
carry.

| Tier | Prompt | Ground truth | What a hit proves |
|---|---|---|---|
| **S1** dictated | "write exactly `Confidence: 85%`" | exact | the readout works; number is in context (copy-level) |
| **S2** forced-extreme | trivial item vs genuinely unanswerable item | ordinal | the number is *computed* -- never in the context |
| **S3** baseline | ~50 MMLU + ~50 TriviaQA | none | the spread the real experiment will see |

**Three failure modes to keep apart**, because they have different consequences:

1. **Format** -- no number emitted at all. Expected on 270m-it; says nothing
   about J-space. The pre-check catches it in seconds.
2. **Degeneracy** -- a number is emitted but never varies. That is R1 / gate V2
   firing early, for pennies instead of 25,000 generations.
3. **Lens** -- the model plainly speaks a number and the lens cannot see it at
   any layer. The only one this notebook exists to answer.

**V1b:** the first digit of S1's dictated number appears in the J-lens top-5 at
some layer in the upper half of the stack, **and** the S2 easy/unanswerable
distributions separate.

In [ ]:
# --- standard header ------------------------------------------------------
%load_ext autoreload
%autoreload 2

import sys, pathlib
for p in ("/workspace/NandaProj", ".."):
    if p not in sys.path:
        sys.path.insert(0, p)

from nandaproj import config

cfg = config.get_model_config()      # NANDA_PRESET env var, defaults to debug
config.ensure_dirs()
print("preset:", cfg.name, "|", cfg.n_params, "|", cfg.dtype)
print("device:", config.get_device())
print("results ->", config.RESULTS)

In [ ]:
def gate(name, ok, detail=""):
    """PLAN.md section 6 verification gate. Fails loudly and stops the notebook."""
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {name}  {detail}")
    if not ok:
        raise AssertionError(f"gate {name} failed: {detail}")

In [ ]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from nandaproj import synthetic, viz

SCALE = "percent"  # "percent" (0-100, multi-token) or "nine" (1-9, single token)
STEP = 5           # candidate grid on the percent scale: 0, 5, ..., 100

tok = AutoTokenizer.from_pretrained(cfg.name, cache_dir=str(config.HF_CACHE))
model = AutoModelForCausalLM.from_pretrained(
    cfg.name,
    cache_dir=str(config.HF_CACHE),
    dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
N_LAYERS = model.config.num_hidden_layers
print(N_LAYERS, "layers,", model.config.hidden_size, "d_model")

## 1. Tokenizer forensics

Everything downstream branches on this. `synthetic.digit_token_ids` raises if any
digit is not a single token -- assuming it is how you silently measure the wrong
thing. On the 0-100 scale it is only half the story: the other half is how many
slots a value costs, which is what the `"85"` / `"100"` / `"85%"` rows below show
directly. Read them as the token budget for every confidence value in the
notebook -- one lens-readable slot, and one or two more that only sequence
scoring can reach.

In [ ]:
for text in ["4", "85", "100", "85%", " 4", "9"]:
    ids = tok.encode(text, add_special_tokens=False)
    print(f"{text!r:>7} -> {ids}  {[tok.decode([i]) for i in ids]}")

DIGIT_IDS = synthetic.digit_token_ids(tok)   # raises if a digit is multi-token
print("\ndigit token ids:", DIGIT_IDS)

CANDS = synthetic.confidence_candidates(step=STEP, scale=SCALE)
print(f"candidates on this scale ({len(CANDS)}):", CANDS)

# How many slots each candidate costs. On percent this is 1-3, and only the
# first is at position -1 -- i.e. only the first is visible to the J-lens.
slots = {len(synthetic.digits_of(c)) for c in CANDS}
print("token slots per candidate:", sorted(slots),
      "-- the lens sees slot 1 only" if slots != {1} else "-- single slot, lens sees all")

## 2. Answer pass, then the confidence slot

Two passes, the shape of PLAN.md's **P1 (separate)** condition: the model commits
to an answer, then rates confidence in an answer already given.

The confidence prompt is **prefilled up to `"Confidence: "`**, so the next token
is the **first digit** of the confidence number, at a known, exact index.
Position `-1` is therefore the answer-adjacent slot by construction -- no
searching, no off-by-one. The remaining digits of a 0-100 value live at `-1+1`,
`-1+2`, which no lens can reach without first committing to what came before;
section 4 walks them.

In [ ]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 24) -> str:
    """Greedy continuation of an already-rendered prompt string."""
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)


@torch.no_grad()
def slot_probs(prompt: str) -> np.ndarray:
    """Final-layer probabilities at the last position -- one forward, one row.

    On 0-100 this row is the distribution over the number's *first digit*, which
    is what the J-lens can see and no more. `conf_dist` is the honest reading of
    the whole value.
    """
    ids = tok(prompt, return_tensors="pt",
              add_special_tokens=False).input_ids.to(model.device)
    return torch.softmax(model(ids).logits[0, -1].float(), dim=-1).cpu().numpy()


def conf_dist(prompt: str) -> np.ndarray:
    """P(confidence == v) for every v in CANDS, as a proper distribution.

    On 0-100 a value spans several tokens, so this sequence-scores `"<v>%"` as a
    continuation: `len(CANDS)` short forward passes per item, not one. That cost
    buys correctness -- an argmax over the single slot would score `8` and call
    it 85, and would rank `100` by its leading `1`.

    On the 1-9 scale a value is one token, so the same quantity is one softmax
    row and the branch below takes it.
    """
    if SCALE == "nine":
        return synthetic.slot_distribution(slot_probs(prompt), DIGIT_IDS, CANDS)
    _, probs = synthetic.score_candidates(model, tok, prompt, CANDS)
    return probs


def answer_of(item, max_new_tokens: int = 24) -> str:
    """First pass: the model's own answer, with any confidence line stripped."""
    rendered = tok.apply_chat_template(
        synthetic.build_chat(item), tokenize=False, add_generation_prompt=True
    )
    text = generate(rendered, max_new_tokens)
    return text.split("Confidence")[0].strip().split("\n")[0]


def confidence_prompt(item) -> str:
    """Full prompt ending exactly at the confidence slot."""
    return synthetic.render(tok, item, answer_of(item))

### Pre-check: does the model say a number at all?

Seconds, and it separates failure mode 1 ("model didn't say a number") from
failure mode 3 ("lens didn't see the number") before any lens work happens. A
`format_ok` of 0 on 270m-it is the expected result, not a bug.

In [34]:
precheck = []
for item in synthetic.all_items(scale=SCALE):
    prompt = confidence_prompt(item)
    said = generate(prompt, max_new_tokens=4)
    probs = slot_probs(prompt)
    precheck.append({
        "item": item.item_id, "tier": item.tier, "target": item.target,
        "said": said.strip()[:12],
        "digit_mass": synthetic.digit_mass(probs, DIGIT_IDS),
    })

for r in precheck:
    print(f"{r['item']:>8}  {r['tier']:<16} said={r['said']!r:<10} "
          f"digit_mass={r['digit_mass']:.3f}")

format_ok = float(np.mean([r["digit_mass"] > 0.5 for r in precheck]))
print(f"\nfraction of items with a digit clearly at the slot: {format_ok:.2f}")
if format_ok < 0.5:
    print("!! FORMAT FAILURE (mode 1). The model is not emitting a confidence "
          "number. Everything below is inconclusive at this scale -- rerun under "
          "NANDA_PRESET=target before reading anything into it.")

  S1_002  S1_dictated      said='2'        digit_mass=1.000
  S1_004  S1_dictated      said='4'        digit_mass=1.000
  S1_006  S1_dictated      said='6'        digit_mass=1.000
  S1_008  S1_dictated      said='8'        digit_mass=1.000
  S2e_00  S2_easy          said='1'        digit_mass=1.000
  S2e_01  S2_easy          said='1'        digit_mass=1.000
  S2e_02  S2_easy          said='1'        digit_mass=1.000
  S2e_03  S2_easy          said='1'        digit_mass=1.000
  S2e_04  S2_easy          said='9'        digit_mass=1.000
  S2u_00  S2_unanswerable  said='9'        digit_mass=1.000
  S2u_01  S2_unanswerable  said='1'        digit_mass=1.000
  S2u_02  S2_unanswerable  said='9'        digit_mass=1.000
  S2u_03  S2_unanswerable  said='1'        digit_mass=1.000
  S2u_04  S2_unanswerable  said='9'        digit_mass=1.000

fraction of items with a digit clearly at the slot: 1.00


### What is actually being asked

Every prompt, laid out before any of it is interpreted. Prompt bugs are the
cheapest way to produce a confident null and they are invisible in a summary
statistic -- a chat template that swallows the instruction, or a prefill landing
one token off, both look exactly like "the lens cannot see it."

`show_prompt` takes a **fixed** answer by default, so you can inspect any item
without a generation pass; pass `answer=None` for the real two-pass prompt.

In [ ]:
import pandas as pd

# Every item, addressable by id: ITEMS["S1_085"], ITEMS["S2u_00"], ...
# The S1 ids carry the target, so they change with the scale: S1_005/037/062/
# 085/100 on percent, S1_002/004/006/008 on nine.
ITEMS = {i.item_id: i for i in synthetic.all_items(scale=SCALE)}

# The item every "one worked example" cell below uses. Index 3 is a two-digit
# target on percent (85) -- deliberately, because a two-digit number is where
# "the lens read the first digit" and "the lens read the number" come apart.
EXAMPLE_ID = synthetic.dictated_items(scale=SCALE)[3].item_id

pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame([{
    "item_id": i.item_id, "tier": i.tier, "scale": i.scale,
    "target": i.target, "question": i.question,
} for i in ITEMS.values()]))

print("\nThe two instructions on this scale, verbatim:")
print("-" * 72)
print(f"S1 dictated ({EXAMPLE_ID}):\n ", ITEMS[EXAMPLE_ID].instruction)
print("\nS2 free:\n ", ITEMS["S2e_00"].instruction)

In [ ]:
def show_prompt(item, answer: str | None = "An answer.", n_tail_tokens: int = 12):
    """Print one item's full prompt and the tokens around the confidence slot.

    `answer` fixed  -> no GPU pass, inspect freely.
    `answer=None`   -> the real two-pass prompt, using the model's own answer.
    """
    if isinstance(item, str):
        item = ITEMS[item]
    ans = answer_of(item) if answer is None else answer
    prompt = synthetic.render(tok, item, ans)

    print(f"=== {item.item_id}  ({item.tier}, scale={item.scale}, target={item.target}) ===")
    print(f"--- answer used: {ans!r} {'(generated)' if answer is None else '(fixed)'}")
    print("--- rendered prompt, repr so the chat template is visible ---")
    print(repr(prompt))
    print("\n--- readable ---")
    print(prompt)

    ids = tok.encode(prompt, add_special_tokens=False)
    tail = ids[-n_tail_tokens:]
    print(f"--- last {len(tail)} tokens (position -1 is the first-digit slot) ---")
    for offset, tid in zip(range(-len(tail), 0), tail):
        print(f"  {offset:>4}  {tid:>7}  {tok.decode([tid])!r}")

    assert not tok.decode([tail[-1]]).strip().isdigit(), (
        "prompt already ends on a digit -- the confidence slot is off by one")
    return prompt


_ = show_prompt(EXAMPLE_ID)

In [ ]:
# The same for an S2 item, where no number is dictated anywhere in the context.
_ = show_prompt("S2u_00")

# Sanity: no S2 prompt may contain a *confidence value* that could be copied.
# Incidental digits are fine and expected ("2 + 2", "14 March 2019") -- what
# would break S2 is a rating sitting in the context, since the readout could
# then be a copy rather than a computation.
#
# Two things this cell used to get wrong, both of which made it vacuous:
#
# 1. `split(CONFIDENCE_PREFIX)[0]` cuts at the FIRST "Confidence: ", which is
#    inside the instruction -- so the context it inspected stopped before the
#    question ever appeared. The questions are the only place a leak could
#    plausibly be. `rsplit(..., 1)` cuts at the prefill slot instead.
# 2. With the whole context in scope the instruction itself now matches, since
#    it legitimately contains both "Confidence" and digits ("a whole number
#    from 0 to 100", or "from 1 ... to 9"). Excise it before grepping, or the
#    check fires on every item on every scale and means nothing.
import re

LEAK = re.compile(r"[Cc]onfidence[^\n]*?\d|\d+\s*%")

for item in synthetic.forced_extreme_items(scale=SCALE):
    context = synthetic.render(tok, item, "An answer.").rsplit(
        synthetic.CONFIDENCE_PREFIX, 1)[0]
    leaked = LEAK.findall(context.replace(item.instruction, ""))
    if leaked:
        print(f"!! {item.item_id} has a copyable rating in context: {leaked}")

# The check has to be able to fail, so prove it can before trusting a clean run.
canary = "Question: pick a number.\nConfidence: 80%"
assert LEAK.findall(canary), "leak regex no longer detects a planted rating"
print("S2 context check done (regex verified against a canary) -- anything "
      "printed above needs fixing before use.")

## 3. S1 - dictated number

The prompt names the number. Copy-level, the same difficulty as V1's `" Judy"`,
and its whole job is to make an S2 null attributable: **if S1 fails, the readout
is broken, not the model.**

On 0-100 the lens reads position `-1`, so what is tested here is the target's
**first digit**: a hit on `85` means the lens surfaced `8`, not `85`. That is a
weaker claim than the 1-9 version made and it is stated as such -- section 4 is
where the second digit is checked, and it is the section that keeps this one
honest.

`use_jacobian=False` gives the vanilla logit lens on the identical activations --
the control that says whether J-space is doing any work, or whether the number
was legible anyway.

In [38]:
import jlens

model_jlens = jlens.from_hf(model, tok)
lens = jlens.JacobianLens.from_pretrained(
    config.LENS_REPO,
    filename=f"{cfg.lens_id}/jlens/Salesforce-wikitext/{cfg.lens_id}_jacobian_lens.pt",
)

# Every layer the lens actually fitted. `lens.source_layers` is the ground
# truth -- `layers=None` in `lens.apply` resolves to exactly this list -- and it
# is NOT always range(n_layers): the 270m lens covers 0-16 of an 18-layer model,
# with no Jacobian for the final layer. Sweeping the full stack costs almost
# nothing (one forward, more decodes) and answers 01's open question about
# which layers have a fitted J_l.
LAYERS = list(lens.source_layers)
missing = sorted(set(range(N_LAYERS)) - set(LAYERS))
print(f"model has {N_LAYERS} layers; lens fitted on {len(LAYERS)}: {LAYERS}")
print("no fitted Jacobian for:", missing or "none")

# The lens is an expectation over corpus text (PLAN.md 4.1). How many prompts
# that average was taken over bounds how much contextual signal it can retain --
# a caveat to quote in the writeup, not a number to discover later.
print(f"lens fitted from n_prompts={lens.n_prompts}, d_model={lens.d_model}")


def readout(prompt, layers=None):
    """Per-layer probability vectors at the last position, both lenses.

    01 called `lens.apply` on a single string; whether it batches is unknown, so
    this loops. Record the answer -- 05/06 budget their runtime on it.
    """
    layers = layers or LAYERS
    jl, ml, _ = lens.apply(model_jlens, prompt, layers=layers, positions=[-1])
    ll, _, _ = lens.apply(model_jlens, prompt, layers=layers, positions=[-1],
                          use_jacobian=False)
    soft = lambda t: torch.softmax(t.float(), dim=-1).cpu().numpy()
    return ({l: soft(jl[l][0]) for l in layers},
            {l: soft(ll[l][0]) for l in layers},
            soft(ml[0]))

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model has 18 layers; lens fitted on 17: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
no fitted Jacobian for: [17]
lens fitted from n_prompts=278, d_model=640


In [ ]:
s1_rows, s1_top5 = [], {}
s1_curves = {"J-lens": [], "logit lens": []}

for item in synthetic.dictated_items(scale=SCALE):
    prompt = confidence_prompt(item)
    j_probs, l_probs, final = readout(prompt)
    # Position -1 holds the *first* digit of the target, and that is all the
    # lens can be asked for. digits_of(85)[0] == 8, digits_of(100)[0] == 1.
    want = synthetic.digits_of(item.target)[0]
    wid = DIGIT_IDS[want]

    s1_curves["J-lens"].append([j_probs[l][wid] for l in LAYERS])
    s1_curves["logit lens"].append([l_probs[l][wid] for l in LAYERS])
    s1_top5[item.item_id] = {
        l: [tok.decode([t]) for t in np.argsort(j_probs[l])[-5:][::-1]] for l in LAYERS
    }
    s1_rows.append({"item": item.item_id, "target": item.target,
                    "first_digit": want, "final_P": float(final[wid])})
    print(f"{item.item_id}  target={item.target:>3}  first digit={want}  "
          f"P(final)={final[wid]:.3f}")

s1_mean = {k: np.mean(np.array(v), axis=0) for k, v in s1_curves.items()}
viz.series_line(
    LAYERS, s1_mean, y_range=(0, 1),
    title="S1 dictated: P(first digit of the dictated number) at the confidence slot",
    xaxis="layer", yaxis="probability",
).show()

In [ ]:
# The example item's prompt exactly as the loop above fed it, immediately before
# the top-5 table is read off it. `answer=None` means the model's own first-pass
# answer, so this is the real two-pass string -- not the fixed-answer version
# section 2 printed. Greedy decoding, so it reproduces what the loop used.
#
# Read the layer table below against this: the number the lens is being asked to
# find is sitting in the instruction, in plain text, a few tokens back -- and
# only its first digit can land at position -1.
_ = show_prompt(EXAMPLE_ID, answer=None)

In [ ]:
# Where in the stack does the dictated number enter the top-5? Across every S1
# item, not just one -- the V1b gate keys on this, and a gate that hinges on a
# single prompt is a coin flip wearing a criterion's clothes.
#
# The match is on the target's FIRST DIGIT, because that is the token at
# position -1. Matching str(item.target) would look for a literal "85" token,
# which the tokenizer never produces, and would report a null on every
# multi-digit target.
def hit_layers(item):
    want = str(synthetic.digits_of(item.target)[0])
    return [l for l in LAYERS
            if want in [t.strip() for t in s1_top5[item.item_id][l]]]


s1_hits = {i.item_id: hit_layers(i) for i in synthetic.dictated_items(scale=SCALE)}
for iid, ls in s1_hits.items():
    print(f"{iid}  first digit in top-5 at layers: {ls or 'never'}")

# One item walked layer by layer, for eyeballing what the lens actually says.
example = ITEMS[EXAMPLE_ID]
print(f"\n--- {example.item_id} (target {example.target}, first digit "
      f"{synthetic.digits_of(example.target)[0]}), J-lens top-5 by layer ---")
for l in LAYERS:
    print(f"L{l:>2}  {s1_top5[example.item_id][l]}")

s1_hit_layers = s1_hits[example.item_id]

# Fraction of S1 items whose dictated first digit is read anywhere in the top
# half of the stack. This is what V1b judges.
s1_hit_frac = float(np.mean(
    [any(l >= N_LAYERS // 2 for l in ls) for ls in s1_hits.values()]))
print(f"\nS1 items read in the top half of the stack: {s1_hit_frac:.0%}")

In [ ]:
# Digit mass and the digit distribution across the stack: "does a number go
# here" (robust) separately from "which digit" (what we actually want). Both are
# read at position -1, so on 0-100 this is the *first* digit of the value.
j_probs, l_probs, final = readout(confidence_prompt(example))

grid = np.array([synthetic.digit_distribution(j_probs[l], DIGIT_IDS) for l in LAYERS])
viz.prob_heatmap(
    grid.T, x=LAYERS, y=list(range(10)),
    title=(f"S1 (target {example.target}): J-lens distribution over the FIRST "
           "digit at the slot"),
    xaxis="layer", yaxis="first digit",
).show()

viz.series_line(
    LAYERS,
    {"J-lens": [synthetic.digit_mass(j_probs[l], DIGIT_IDS) for l in LAYERS],
     "logit lens": [synthetic.digit_mass(l_probs[l], DIGIT_IDS) for l in LAYERS]},
    y_range=(0, 1),
    title="Digit mass: where 'a number goes here' crystallizes",
    xaxis="layer", yaxis="P(token is a digit)",
).show()

## 4. The whole number, not just its first digit

Section 3 tested one slot. On 0-100 that is the *first* digit, and **only**
checking the first digit is not enough: a model that says "8" then anything at
all would score as a hit on 85.

Each digit sits at its own slot. Digit 2 is read by appending digit 1 to the
prompt and re-running, which is exactly what generation does. The joint
probability -- the product over the digits -- is the number that matters, and it
is always <= the first-digit figure. The gap between the two columns below is
precisely how much section 3's headline overstates the readout.

This section ran on `scale="percent"` even when the rest of the notebook was on
1-9. Now it is the same scale as everything above, so it is a direct
continuation of section 3 rather than a side experiment.

In [ ]:
pct_rows = []
for item in synthetic.dictated_items(scale="percent"):
    want = synthetic.digits_of(item.target)
    base = confidence_prompt(item)

    row = {"item": item.item_id, "target": item.target, "digits": want}
    prompt = base
    for pos, d in enumerate(want):
        j_probs, l_probs, final = readout(prompt)
        top_layer = LAYERS[-1]
        row[f"d{pos}"] = d
        row[f"d{pos}_final"] = float(final[DIGIT_IDS[d]])
        row[f"d{pos}_jlens_L{top_layer}"] = float(j_probs[top_layer][DIGIT_IDS[d]])
        prompt = prompt + str(d)      # advance one slot, as generation would
    pct_rows.append(row)

pct = pd.DataFrame(pct_rows)
display(pct)

# Joint probability of the whole number: the product over its digits. This is
# the number that matters, and it is always <= the first-digit figure.
joint = [np.prod([r[f"d{p}_final"] for p in range(len(r["digits"]))]) for r in pct_rows]
for r, j in zip(pct_rows, joint):
    firsts = r["d0_final"]
    print(f"{r['target']:>4}: P(first digit)={firsts:.3f}  P(whole number)={j:.3f}"
          f"   {'<-- first digit alone overstates it' if firsts > 2 * j else ''}")

# The same comparison for the J-lens at the top fitted layer. Section 3's
# headline is the d0 column; this is what it is worth once the rest of the
# number has to be right too.
j_joint = [np.prod([r[f"d{p}_jlens_L{LAYERS[-1]}"] for p in range(len(r["digits"]))])
           for r in pct_rows]
print(f"\nJ-lens at L{LAYERS[-1]}, first digit vs whole number:")
for r, j in zip(pct_rows, j_joint):
    print(f"{r['target']:>4}: {r[f'd0_jlens_L{LAYERS[-1]}']:.3f} -> {j:.3f}")

### Control: the V1 prompt, unchanged

Before asking whether the lens can see a *computed* number, check that it can
still see the thing it already saw. This is V1's prompt verbatim -- `" Judy"`,
a name sitting in plain sight two clauses back -- read through the same `lens`
object, the same `readout`, and the same layer list everything above used.

It is **raw text: no chat template, no prefilled `"Confidence: "`, no answer
pass.** That is the point. It separates two failures that look identical in a
summary statistic:

- Judy reads, digits do not -> the instrument works and the *number* is the
  hard part. That is the real finding, and section 5 is where it gets tested.
- Judy does not read either -> the lens, the wrapper, or the layer list is
  broken, and nothing above means anything. Fix that before reading a single
  confidence curve.

Cheap enough to leave in: one prompt, two forward passes.


In [43]:
# 01's exact prompt, through this notebook's `readout` and this notebook's lens.
JUDY = "Jake and Judy were talking to each other. Jake then handed his toy to"

j_probs, l_probs, final = readout(JUDY)


def top5(probs):
    return [tok.decode([t]) for t in np.argsort(probs)[-5:][::-1]]


print(f"{JUDY!r}\n")
print(f"{'layer':>5}  {'J-lens top-5':<46}  logit lens top-5")
for l in LAYERS:
    print(f"{l:>5}  {str(top5(j_probs[l])):<46}  {top5(l_probs[l])}")
print(f"\nmodel's own next token: {top5(final)}")

judy_layers = [l for l in LAYERS if "Judy" in [t.strip() for t in top5(j_probs[l])]]
upper = [l for l in judy_layers if l >= N_LAYERS // 2]
print(f"\n'Judy' in the J-lens top-5 at layers: {judy_layers or 'never'}")

# A control, not a PLAN.md gate -- but it stops the notebook, because every
# reading below is uninterpretable if the lens cannot do the thing it already
# did in 01.
gate("control: V1 Judy readout still works",
     bool(upper),
     f"upper-half layers with Judy: {upper or 'none'} "
     f"(01 saw it from L12 up on 270m-it)")


'Jake and Judy were talking to each other. Jake then handed his toy to'

layer  J-lens top-5                                    logit lens top-5
    0  [' Rome', ' Ro', ' wanna', ' Toc', ' capitol']  [' zu', ' to', 'to', ' TO', ' zur']
    1  [' hereof', 'ด้อ', 'foresaid', 'agory', ' seater']  ['setPrototypeOf', 'idän', '</h5>', '〟', '冋']
    2  ['ด้อ', ' everybody', 'agory', ' wanna', ' guy']  ['ккей', 'idän', 'setPrototypeOf', 'Ћ', 'новні']
    3  ['oler', 'ด้อ', ' lordship', 'belly', ' hereof']  ['ккей', 'idän', 'ప్పటికీ', 'тьяна', 'setPrototypeOf']
    4  ['oler', ' margarine', 'belly', ' piglets', 'fila']  ['ккей', 'setPrototypeOf', 'ప్పటికీ', '../../', 'новні']
    5  [' margarine', 'iteracy', ' piglets', ' literacy', 'oler']  ['fitrión', 'новні', ' og', 'amanho', 'idän']
    6  [' inanimate', ' toddlers', ' turnips', ' babies', ' toddler']  ['idän', 'fitrión', 'habit', ' fhe', ' og']
    7  [' kids', ' kiddos', ' inanimate', ' kid', ' lefty']  ['idän', ' việc', 'fitrión', ' fhe'

## 5. S2 - forced extreme

The number is **never in the context**; it has to be computed. Ground truth is
ordinal: easy should exceed unanswerable. The hard arm is *genuinely*
unanswerable rather than merely difficult -- overconfidence on hard questions is
the known phenomenon, so hard-but-answerable would confound the test. If
confidence stays high even here, that is a strong standalone observation.

On 0-100 the distribution below is a **sequence score over 21 candidates**
(`conf_dist`), not one softmax row. So it is not the same object the lens reads:
the lens sees the first digit at position `-1`, this sees the whole value. The
two agree on ordering but they are different measurements, and the second cell
is careful to say which one it is plotting.

In [ ]:
from collections import defaultdict

# CANDS came from section 1. Each item costs len(CANDS) short forward passes on
# the percent scale -- 10 items, so ~210 passes. Cheap here; section 6 is where
# it starts to matter.
s2_dists, s2_expected = defaultdict(list), defaultdict(list)
s2_by_item = {}

for item in synthetic.forced_extreme_items(scale=SCALE):
    dist = conf_dist(confidence_prompt(item))
    s2_by_item[item.item_id] = dist
    s2_dists[item.tier].append(dist)
    s2_expected[item.tier].append(synthetic.expected_confidence(CANDS, dist))
    print(f"{item.item_id}  {item.tier:<16} E[conf]="
          f"{s2_expected[item.tier][-1]:5.1f}/100  "
          f"mode={CANDS[int(np.argmax(dist))]:>3}   {item.question[:40]!r}")

easy = float(np.mean(s2_expected["S2_easy"]))
hard = float(np.mean(s2_expected["S2_unanswerable"]))
print(f"\nmean E[confidence]  easy={easy:.1f}  unanswerable={hard:.1f}  "
      f"separation={easy - hard:+.1f} points on a 0-100 scale")

viz.grouped_bar(
    CANDS,
    {"easy": np.mean(np.array(s2_dists["S2_easy"]), axis=0),
     "unanswerable": np.mean(np.array(s2_dists["S2_unanswerable"]), axis=0)},
    title="S2: confidence distribution over the whole value (sequence-scored)",
    xaxis=f"stated confidence (0-100, step {STEP})", yaxis="probability",
).show()

In [ ]:
# Does the *computed* number show up in J-space before it is spoken? Track the
# first digit of the value the model actually commits to, per arm.
#
# "Commits to" is the mode of the sequence-scored distribution from the cell
# above (reused, not recomputed -- it cost 21 forwards per item). Its first
# digit is the only part of it that lives at position -1, which is the only
# position the lens reads.
s2_curves = {"J-lens": [], "logit lens": []}
for item in synthetic.forced_extreme_items(scale=SCALE):
    j_probs, l_probs, final = readout(confidence_prompt(item))
    spoken = CANDS[int(np.argmax(s2_by_item[item.item_id]))]
    sid = DIGIT_IDS[synthetic.digits_of(spoken)[0]]
    s2_curves["J-lens"].append([j_probs[l][sid] for l in LAYERS])
    s2_curves["logit lens"].append([l_probs[l][sid] for l in LAYERS])

viz.series_line(
    LAYERS, {k: np.mean(np.array(v), axis=0) for k, v in s2_curves.items()},
    y_range=(0, 1),
    title="S2 computed: P(first digit of the value the model goes on to say)",
    xaxis="layer", yaxis="probability",
).show()

## 6. S3 - dataset baseline

~50 MMLU + ~50 TriviaQA items through the same readout, to see the spread the
real experiment will be working with. Qualitative: it exercises the `data.py`
loaders and gives the contrast against the synthetic tiers. It does **not**
collect activations -- that is 02's job, under the fixed P1-P5 grid.

In [58]:
from nandaproj import data

N_BASELINE = 50
mmlu = data.load_dataset_mmlu(n_items=N_BASELINE)
tqa = data.load_dataset_triviaqa(n_items=N_BASELINE)
print(f"MMLU {len(mmlu['item_ids'])} items | TriviaQA {len(tqa['item_ids'])} items")
print("MMLU sample:", mmlu["questions"][0][:80], "|", mmlu["options"][0])
print("TQA  sample:", tqa["questions"][0][:80], "|", tqa["answers_text"][0])

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

MMLU 50 items | TriviaQA 50 items
MMLU sample: The cyclic subgroup of Z_24 generated by 18 has order | ['4', '8', '12', '6']
TQA  sample: Who was the man behind The Chipmunks? | David Seville


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

MMLU 50 items | TriviaQA 50 items
MMLU sample: The cyclic subgroup of Z_24 generated by 18 has order | ['4', '8', '12', '6']
TQA  sample: Who was the man behind The Chipmunks? | David Seville


In [ ]:
from tqdm.auto import tqdm


def baseline_items(ds, kind):
    """Wrap dataset rows as SyntheticItems so they run the identical path.

    The tier label is S2_easy purely so the dataclass validates; nothing here is
    synthetic and no ground-truth confidence is claimed.
    """
    out = []
    for i, q in enumerate(ds["questions"]):
        if kind == "mmlu":
            a, b, c, d = ds["options"][i]
            q = f"{q}\nA. {a}\nB. {b}\nC. {c}\nD. {d}"
        out.append(synthetic.SyntheticItem(ds["item_ids"][i], "S2_easy", q, scale=SCALE))
    return out


# On percent this is the expensive cell in the notebook: one generation pass per
# item for the answer, then len(CANDS) scoring passes for the distribution --
# 2*N_BASELINE * (1 + 21) forwards, not 2*N_BASELINE. Hence the progress bar;
# at `target` this is minutes, not seconds.
baseline = {}
for kind, ds in (("mmlu", mmlu), ("triviaqa", tqa)):
    vals = []
    for item in tqdm(baseline_items(ds, kind), desc=f"S3 {kind}"):
        vals.append(synthetic.expected_confidence(
            CANDS, conf_dist(confidence_prompt(item))))
    baseline[kind] = np.array(vals)
    print(f"{kind:>9}  mean={np.mean(vals):5.1f}  sd={np.std(vals):5.1f}  "
          f"min={np.min(vals):5.1f}  max={np.max(vals):5.1f}")

# Bins centred on the candidate grid, so a bar means "items whose E[confidence]
# rounds to this candidate" rather than an arbitrary slicing of 0-100.
half = STEP / 2
viz.grouped_bar(
    CANDS,
    {k: np.histogram(v, bins=len(CANDS), range=(-half, 100 + half))[0] / len(v)
     for k, v in baseline.items()},
    title="S3 baseline: spread of E[verbalized confidence] over real items",
    xaxis=f"stated confidence (0-100, step {STEP})", yaxis="fraction of items",
).show()

### The V2 preview

The spread above is an early look at gate **V2** (PLAN.md 6, R1): if `sd` is near
zero on both datasets, verbalized confidence is degenerate at this scale and the
phenomenon does not exist here. That is R1 firing at hour 4 on 100 items instead
of hour 6 on 25,000 -- escalate rather than collect.

Escalation note: `neuronpedia/jacobian-lens` has fitted lenses for
`llama3.1-8b-it` and `qwen2.5-7b-it`, the models of 2603.25052. Those give a lens
*and* published confidence-gap numbers to check the 5.2 replication floor
against, which `gemma-3-12b-it` does not.

In [ ]:
# V1b. Both halves must hold: the readout works (S1), and the computed number
# separates in the right direction (S2).
#
# S1 is judged across every dictated item. With five items and ten digits, one
# lucky top-5 would otherwise carry the whole gate.
#
# On 0-100 the S1 half is a FIRST-DIGIT criterion, which is weaker than the
# single-token version. Section 4's whole-number column is the qualifier to
# carry into the writeup alongside this PASS.
s1_ok = s1_hit_frac >= 0.75
s2_separates = easy > hard

gate("V1b lens reads a confidence number",
     bool(s1_ok and s2_separates),
     f"S1 first digit read in top half: {s1_hit_frac:.0%} of items (need >=75%) | "
     f"S2 easy={easy:.1f} unanswerable={hard:.1f} on 0-100 | "
     f"format_ok={format_ok:.2f}")

### Record before moving on

- Which of the three failure modes fired, and at which preset?
- Does `lens.apply` batch, or is the loop in `readout` forced? 05/06 budget on it.
- The layer at which the dictated first digit first enters the J-lens top-5 --
  the natural centre for 05's sweep.
- `sd` of E[confidence] on MMLU and TriviaQA -> the V2 preview.
- **How much section 3's first-digit result shrinks in section 4** once the whole
  number has to be right. That gap is the honest size of the readout claim on a
  0-100 scale, and it decides whether 02 stays on percent or moves to the
  single-token 1-9 scale (`SCALE = "nine"`, everything below it still runs).
- Wall-clock for the whole notebook at `target`, for the hours 4-8 budget --
  section 6 now costs ~22 forwards per item rather than 2.